In [ ]:
!pip install implicit pandas numpy scipy scikit-learn tqdm

In [ ]:
import os
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix

from implicit.als import AlternatingLeastSquares

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

from tqdm import tqdm

In [ ]:
DATA_ZIP = Path("submissions_archive_2026-05-14.zip")
DATA_DIR = Path("submissions_export")

OUTPUT_DIR = Path("hybrid_recommender_output")
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42

Достаем данные

In [ ]:
def extract_if_needed(zip_path, data_dir):
    if data_dir.exists() and any(data_dir.glob("*.csv")):
        print(f"Папка уже есть: {data_dir}")
        return

    if not zip_path.exists():
        raise FileNotFoundError(f"Не найден архив: {zip_path}")

    data_dir.mkdir(exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(data_dir)

    print(f"Архив распакован в: {data_dir}")


def read_export_csv(file_name):
    path = DATA_DIR / file_name

    if not path.exists():
        raise FileNotFoundError(f"Не найден файл: {path}")

    return pd.read_csv(path, sep=";", encoding="utf-8-sig")


extract_if_needed(DATA_ZIP, DATA_DIR)

user_comp_df = read_export_csv("user_competences.csv")
task_sub_df = read_export_csv("user_task_submissions.csv")
practice_df = read_export_csv("user_practice_submissions.csv")
task_comp_df = read_export_csv("task_competence_mapping.csv")
micro_df = read_export_csv("micromodules_competencies.csv")

Папка уже есть: submissions_export


In [ ]:
print("user_comp_df:", user_comp_df.shape)
print("task_sub_df:", task_sub_df.shape)
print("practice_df:", practice_df.shape)
print("task_comp_df:", task_comp_df.shape)
print("micro_df:", micro_df.shape)


user_comp_df: (7100, 11)
task_sub_df: (2766, 17)
practice_df: (325, 12)
task_comp_df: (1613, 10)
micro_df: (1038, 11)


Берем последнее актуальное состояние user x comp

In [ ]:
def prepare_user_competences(df):
    uc = df.copy()

    uc["user_id"] = uc["user_id"].astype(str)
    uc["competence_id"] = uc["competence_id"].astype(str)

    if "competence_code" in uc.columns:
        uc["competence_code"] = uc["competence_code"].astype(str)

    if "competence_name" in uc.columns:
        uc["competence_name"] = uc["competence_name"].astype(str)

    uc["probability"] = pd.to_numeric(
        uc["probability"],
        errors="coerce"
    )

    uc["evidence_count"] = pd.to_numeric(
        uc["evidence_count"],
        errors="coerce"
    ).fillna(0)

    uc["assessed_at"] = pd.to_datetime(
        uc["assessed_at"],
        errors="coerce"
    )

    uc = uc.dropna(
        subset=["user_id", "competence_id", "probability"]
    )

    uc["probability"] = uc["probability"].clip(0, 1)

    uc = uc.sort_values(
        ["user_id", "competence_id", "evidence_count", "assessed_at"],
        ascending=[True, True, True, True]
    )

    uc_latest = (
        uc
        .groupby(["user_id", "competence_id"], as_index=False)
        .tail(1)
        .reset_index(drop=True)
    )

    uc_latest["mastery"] = uc_latest["probability"]

    uc_latest["need"] = 1 - uc_latest["mastery"]

    return uc_latest

Актуальная таблица UserCompetence

In [ ]:
uc_latest = prepare_user_competences(user_comp_df)

print("Было строк:", user_comp_df.shape)
print("Стало строк:", uc_latest.shape)

print("Users:", uc_latest["user_id"].nunique())
print("Competences:", uc_latest["competence_id"].nunique())

display(uc_latest.head())

Было строк: (7100, 11)
Стало строк: (971, 13)
Users: 30
Competences: 92


,user_id,user_number,competence_id,competence_code,competence_name,probability,assessment_source,evidence_count,assessed_at,valid_from,valid_to,mastery,need
0,04faac8a-55ba-45e8-b22b-3defa5512390,1,078423e2-9dcd-4e8e-9887-702ee08b0153,6.2.2,Умеет приводить подобные слагаемые и упрощать ...,0.956498,calculated,6,2026-03-16 18:27:31.277411,2026-03-16 18:27:31.277411,NaN,0.956498,0.043502
1,04faac8a-55ba-45e8-b22b-3defa5512390,1,1024e3ec-6786-4059-84fe-3ae001527444,7.0.0,Вычисления и преобразования,0.035088,calculated,2,2026-03-16 18:27:31.277411,2026-03-16 18:27:31.277411,NaN,0.035088,0.964912
2,04faac8a-55ba-45e8-b22b-3defa5512390,1,118d8585-a00f-46f7-be45-0a39fc33f401,6.16.1,Умеет представлять числа и дроби в виде степен...,0.115756,calculated,2,2026-03-16 18:27:31.277411,2026-03-16 18:27:31.277411,NaN,0.115756,0.884244
3,04faac8a-55ba-45e8-b22b-3defa5512390,1,2e7f87eb-5a14-4b20-99f4-7efc507e829b,4.1.4,"Умеет вычислять значение отношения двух чисел,...",0.629160,calculated,4,2026-03-16 18:27:31.277411,2026-03-16 18:27:31.277411,NaN,0.629160,0.370840
4,04faac8a-55ba-45e8-b22b-3defa5512390,1,2f810ad4-1d9a-47a2-ac2a-54f8d6c46692,6.14.1,Умеет интерпретировать корень нечётной степени...,0.035088,calculated,1,2026-03-15 18:13:54.078542,2026-03-15 18:13:54.078542,NaN,0.035088,0.964912


Подготовка опыта пользователя по задачам

* сколько задач решал;
* сколько правильно;
* по каким компетенциям были попытки;
* какой средний score;
* когда была последняя попытка.

In [ ]:
def prepare_task_experience(task_submissions, task_competences):
    ts = task_submissions.copy()
    tc = task_competences.copy()

    ts["user_id"] = ts["user_id"].astype(str)
    ts["task_id"] = ts["task_id"].astype(str)

    tc["task_id"] = tc["task_id"].astype(str)
    tc["competence_id"] = tc["competence_id"].astype(str)

    if "is_correct" in ts.columns:
        ts["is_correct_num"] = ts["is_correct"].astype(bool).astype(int)
    else:
        ts["is_correct_num"] = 0

    ts["score"] = pd.to_numeric(
        ts.get("score", 0),
        errors="coerce"
    ).fillna(0)

    ts["max_score"] = pd.to_numeric(
        ts.get("max_score", 1),
        errors="coerce"
    )

    ts["max_score"] = ts["max_score"].replace(0, np.nan)

    ts["score_share"] = (
        ts["score"] / ts["max_score"]
    ).replace([np.inf, -np.inf], np.nan).fillna(0).clip(0, 1)

    ts["submission_time"] = pd.to_datetime(
        ts.get("submission_time"),
        errors="coerce",
        utc=True
    )

    tc_small = tc[["task_id", "competence_id"]].drop_duplicates()

    merged = ts.merge(
        tc_small,
        on="task_id",
        how="inner"
    )

    exp = (
        merged
        .groupby(["user_id", "competence_id"], as_index=False)
        .agg(
            task_attempts=("task_id", "count"),
            task_correct_attempts=("is_correct_num", "sum"),
            task_correct_rate=("is_correct_num", "mean"),
            task_avg_score=("score_share", "mean"),
            last_task_time=("submission_time", "max")
        )
    )

    return exp

In [ ]:
task_exp = prepare_task_experience(
    task_sub_df,
    task_comp_df
)

print("task_exp:", task_exp.shape)
display(task_exp.head())

task_exp: (1261, 7)


,user_id,competence_id,task_attempts,task_correct_attempts,task_correct_rate,task_avg_score,last_task_time
0,04faac8a-55ba-45e8-b22b-3defa5512390,078423e2-9dcd-4e8e-9887-702ee08b0153,6,6,1.0,1.0,2026-03-16 18:27:29.926178+00:00
1,04faac8a-55ba-45e8-b22b-3defa5512390,1024e3ec-6786-4059-84fe-3ae001527444,2,1,0.5,0.5,2026-03-16 18:27:29.926178+00:00
2,04faac8a-55ba-45e8-b22b-3defa5512390,118d8585-a00f-46f7-be45-0a39fc33f401,2,2,1.0,1.0,2026-03-16 18:27:29.926178+00:00
3,04faac8a-55ba-45e8-b22b-3defa5512390,2e7f87eb-5a14-4b20-99f4-7efc507e829b,4,4,1.0,1.0,2026-03-16 18:27:29.926178+00:00
4,04faac8a-55ba-45e8-b22b-3defa5512390,2f810ad4-1d9a-47a2-ac2a-54f8d6c46692,1,1,1.0,1.0,2026-03-15 18:13:52.861923+00:00


Общий профиль пользователя

По ней можно понять:

* сколько всего задач решил пользователь;
* какой у него correct rate;
* сколько было практик;
* сколько было КИМов;
* сколько уникальных задач и микромодулей он трогал.

In [ ]:
def prepare_user_summary(task_submissions, practice_submissions):
    ts = task_submissions.copy()
    pr = practice_submissions.copy()

    ts["user_id"] = ts["user_id"].astype(str)
    pr["user_id"] = pr["user_id"].astype(str)

    ts["is_correct_num"] = ts["is_correct"].astype(bool).astype(int)

    ts["submission_time"] = pd.to_datetime(
        ts.get("submission_time"),
        errors="coerce",
        utc=True
    )

    task_summary = (
        ts
        .groupby("user_id", as_index=False)
        .agg(
            solved_tasks=("task_id", "count"),
            unique_tasks=("task_id", "nunique"),
            unique_nodes=("node_id", "nunique"),
            task_correct_rate=("is_correct_num", "mean"),
            first_task_time=("submission_time", "min"),
            last_task_time=("submission_time", "max")
        )
    )

    pr["submission_time"] = pd.to_datetime(
        pr.get("submission_time"),
        errors="coerce",
        utc=True
    )

    pr["type"] = pr["type"].fillna("UNKNOWN")

    practice_summary = (
        pr
        .groupby("user_id", as_index=False)
        .agg(
            total_practices=("practice_submission_id", "count"),
            kim_count=("type", lambda x: (x == "KIM").sum()),
            block_count=("type", lambda x: (x != "KIM").sum()),
            avg_test_score=("test_score", "mean"),
            avg_total_score=("total_score", "mean"),
            first_practice_time=("submission_time", "min"),
            last_practice_time=("submission_time", "max")
        )
    )

    user_summary = task_summary.merge(
        practice_summary,
        on="user_id",
        how="outer"
    )

    return user_summary

In [ ]:
user_summary = prepare_user_summary(
    task_sub_df,
    practice_df
)

print("user_summary:", user_summary.shape)
display(user_summary.head())

user_summary: (38, 14)


,user_id,solved_tasks,unique_tasks,unique_nodes,task_correct_rate,first_task_time,last_task_time,total_practices,kim_count,block_count,avg_test_score,avg_total_score,first_practice_time,last_practice_time
0,04faac8a-55ba-45e8-b22b-3defa5512390,64.0,58.0,17.0,0.953125,2026-02-25 17:41:51.998349+00:00,2026-03-22 09:40:01.786033+00:00,9,4,5,38.555556,6.777778,2026-02-25 17:41:51.998349+00:00,2026-03-22 09:40:01.786033+00:00
1,0e24ff0e-c58a-4a6c-b054-e80df6607b99,14.0,14.0,14.0,0.785714,2026-03-11 20:18:02.450819+00:00,2026-03-11 20:18:02.450819+00:00,1,1,0,76.000000,15.000000,2026-03-11 20:18:02.450819+00:00,2026-03-11 20:18:02.450819+00:00
2,17993c65-1f0e-4f0a-b458-a7c688b99ea0,212.0,177.0,46.0,0.622642,2026-02-20 06:17:00.025055+00:00,2026-04-09 16:45:52.223790+00:00,19,10,9,39.052632,6.947368,2026-02-20 06:17:00.025055+00:00,2026-04-09 16:45:52.223790+00:00
3,2313e3e0-a834-4dfd-a62e-da6bf6bab116,9.0,9.0,8.0,0.222222,NaT,NaT,3,3,0,3.666667,0.666667,2026-01-29 17:26:14.122537+00:00,2026-01-29 17:39:01.419094+00:00
4,363f7164-3a52-43aa-8b9b-b1677ed218ec,216.0,184.0,55.0,0.750000,2026-02-22 14:49:19.246889+00:00,2026-04-05 07:56:28.460589+00:00,17,5,12,60.352941,11.176471,2026-02-22 14:49:19.246889+00:00,2026-04-05 07:56:28.460589+00:00


Текстовое описание компетенции

In [ ]:
def build_competence_catalog(uc_latest, micro_df, task_comp_df):
    base_frames = []

    for df in [uc_latest, micro_df, task_comp_df]:
        cols = [
            col for col in [
                "competence_id",
                "competence_code",
                "competence_name"
            ]
            if col in df.columns
        ]

        if "competence_id" in cols:
            base_frames.append(df[cols].drop_duplicates())

    catalog = (
        pd.concat(base_frames, ignore_index=True)
        .drop_duplicates(subset=["competence_id"])
        .reset_index(drop=True)
    )

    catalog["competence_id"] = catalog["competence_id"].astype(str)

    if "competence_code" not in catalog.columns:
        catalog["competence_code"] = ""

    if "competence_name" not in catalog.columns:
        catalog["competence_name"] = ""

    catalog["competence_code"] = (
        catalog["competence_code"]
        .fillna("")
        .astype(str)
    )

    catalog["competence_name"] = (
        catalog["competence_name"]
        .fillna("")
        .astype(str)
    )

    catalog["competence_text"] = catalog["competence_name"]

    empty_mask = catalog["competence_text"].str.strip() == ""

    catalog.loc[empty_mask, "competence_text"] = catalog.loc[
        empty_mask,
        "competence_code"
    ]

    empty_mask = catalog["competence_text"].str.strip() == ""

    catalog.loc[empty_mask, "competence_text"] = catalog.loc[
        empty_mask,
        "competence_id"
    ]

    return catalog

In [ ]:
comp_catalog = build_competence_catalog(
    uc_latest,
    micro_df,
    task_comp_df
)

print("comp_catalog:", comp_catalog.shape)
display(comp_catalog.head())

comp_catalog: (225, 4)


,competence_id,competence_code,competence_name,competence_text
0,078423e2-9dcd-4e8e-9887-702ee08b0153,6.2.2,Умеет приводить подобные слагаемые и упрощать ...,Умеет приводить подобные слагаемые и упрощать ...
1,1024e3ec-6786-4059-84fe-3ae001527444,7.0.0,Вычисления и преобразования,Вычисления и преобразования
2,118d8585-a00f-46f7-be45-0a39fc33f401,6.16.1,Умеет представлять числа и дроби в виде степен...,Умеет представлять числа и дроби в виде степен...
3,2e7f87eb-5a14-4b20-99f4-7efc507e829b,4.1.4,"Умеет вычислять значение отношения двух чисел,...","Умеет вычислять значение отношения двух чисел,..."
4,2f810ad4-1d9a-47a2-ac2a-54f8d6c46692,6.14.1,Умеет интерпретировать корень нечётной степени...,Умеет интерпретировать корень нечётной степени...


Объединяем UserCompetence и опыт пользователя

* probability — насколько пользователь владеет компетенцией;
* need — насколько её нужно подтянуть;
* evidence_count — сколько было касаний;
* task_attempts — сколько задач по этой компетенции решал пользователь;
* als_value — итоговый сигнал для ALS.

Формула для ALS:

als_value = (0.05 + need) × (1 + log(1 + evidence_count + task_attempts))

In [ ]:
def build_model_table(uc_latest, task_exp):
    df = uc_latest.copy()

    df = df.merge(
        task_exp,
        on=["user_id", "competence_id"],
        how="left"
    )

    for col in [
        "task_attempts",
        "task_correct_attempts",
        "task_correct_rate",
        "task_avg_score"
    ]:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        ).fillna(0)

    df["total_experience"] = (
        df["evidence_count"].fillna(0)
        + df["task_attempts"].fillna(0)
    )

    df["experience_weight"] = np.log1p(
        df["total_experience"]
    )

    df["need"] = (
        1 - df["probability"].clip(0, 1)
    ).clip(0, 1)

    df["als_value"] = (
        (0.05 + df["need"])
        * (1 + df["experience_weight"])
    )

    df["als_value"] = pd.to_numeric(
        df["als_value"],
        errors="coerce"
    ).fillna(0)

    df = df[df["als_value"] > 0].copy()

    return df

In [ ]:
model_df = build_model_table(
    uc_latest,
    task_exp
)

print("model_df:", model_df.shape)

print("Users:", model_df["user_id"].nunique())
print("Competences:", model_df["competence_id"].nunique())

display(model_df.head())

model_df: (971, 21)
Users: 30
Competences: 92


,user_id,user_number,competence_id,competence_code,competence_name,probability,assessment_source,evidence_count,assessed_at,valid_from,...,mastery,need,task_attempts,task_correct_attempts,task_correct_rate,task_avg_score,last_task_time,total_experience,experience_weight,als_value
0,04faac8a-55ba-45e8-b22b-3defa5512390,1,078423e2-9dcd-4e8e-9887-702ee08b0153,6.2.2,Умеет приводить подобные слагаемые и упрощать ...,0.956498,calculated,6,2026-03-16 18:27:31.277411,2026-03-16 18:27:31.277411,...,0.956498,0.043502,6.0,6.0,1.0,1.0,2026-03-16 18:27:29.926178+00:00,12.0,2.564949,0.333328
1,04faac8a-55ba-45e8-b22b-3defa5512390,1,1024e3ec-6786-4059-84fe-3ae001527444,7.0.0,Вычисления и преобразования,0.035088,calculated,2,2026-03-16 18:27:31.277411,2026-03-16 18:27:31.277411,...,0.035088,0.964912,2.0,1.0,0.5,0.5,2026-03-16 18:27:29.926178+00:00,4.0,1.609438,2.648351
2,04faac8a-55ba-45e8-b22b-3defa5512390,1,118d8585-a00f-46f7-be45-0a39fc33f401,6.16.1,Умеет представлять числа и дроби в виде степен...,0.115756,calculated,2,2026-03-16 18:27:31.277411,2026-03-16 18:27:31.277411,...,0.115756,0.884244,2.0,2.0,1.0,1.0,2026-03-16 18:27:29.926178+00:00,4.0,1.609438,2.437853
3,04faac8a-55ba-45e8-b22b-3defa5512390,1,2e7f87eb-5a14-4b20-99f4-7efc507e829b,4.1.4,"Умеет вычислять значение отношения двух чисел,...",0.629160,calculated,4,2026-03-16 18:27:31.277411,2026-03-16 18:27:31.277411,...,0.629160,0.370840,4.0,4.0,1.0,1.0,2026-03-16 18:27:29.926178+00:00,8.0,2.197225,1.345520
4,04faac8a-55ba-45e8-b22b-3defa5512390,1,2f810ad4-1d9a-47a2-ac2a-54f8d6c46692,6.14.1,Умеет интерпретировать корень нечётной степени...,0.035088,calculated,1,2026-03-15 18:13:54.078542,2026-03-15 18:13:54.078542,...,0.035088,0.964912,1.0,1.0,1.0,1.0,2026-03-15 18:13:52.861923+00:00,2.0,1.098612,2.129907


Строим матрицу user × competence для als

In [ ]:
def build_sparse_matrix(df, value_col):
    users = df["user_id"].astype("category")
    comps = df["competence_id"].astype("category")

    user_index = users.cat.categories.astype(str)
    comp_index = comps.cat.categories.astype(str)

    rows = users.cat.codes
    cols = comps.cat.codes

    values = df[value_col].astype(float).values

    matrix = csr_matrix(
        (
            values,
            (rows, cols)
        ),
        shape=(len(user_index), len(comp_index))
    )

    return matrix, user_index, comp_index

In [ ]:
als_matrix, user_index, comp_index = build_sparse_matrix(
    model_df,
    "als_value"
)

print("als_matrix:", als_matrix.shape)
print("non-zero:", als_matrix.nnz)

sparsity = 1 - als_matrix.nnz / (
    als_matrix.shape[0] * als_matrix.shape[1]
)

print(f"sparsity: {sparsity:.2%}")

als_matrix: (30, 92)
non-zero: 971
sparsity: 64.82%


Обучение ALS

ALS отвечает за опыт похожих пользователей.

In [ ]:
ALS_FACTORS = 32
ALS_REGULARIZATION = 0.1
ALS_ITERATIONS = 50
ALS_ALPHA = 20.0

In [ ]:
def train_als(matrix):
    model = AlternatingLeastSquares(
        factors=ALS_FACTORS,
        regularization=ALS_REGULARIZATION,
        iterations=ALS_ITERATIONS,
        random_state=RANDOM_STATE
    )

    model.fit(
        (matrix * ALS_ALPHA).tocsr()
    )

    return model

In [ ]:
als_model = train_als(als_matrix)

print("ALS обучен")
print("user_factors:", als_model.user_factors.shape)
print("item_factors:", als_model.item_factors.shape)

  0%|          | 0/50 [00:00<?, ?it/s]

ALS обучен
user_factors: (30, 32)
item_factors: (92, 32)


Текстовые векторы компетенций

Теперь добавляем вторую часть рекомендательной системы — текстовую.

Каждая компетенция превращается в TF-IDF-вектор по её текстовому описанию.

In [ ]:
item_catalog = pd.DataFrame({
    "competence_id": comp_index.astype(str)
})

item_catalog = item_catalog.merge(
    comp_catalog,
    on="competence_id",
    how="left"
)

item_catalog["competence_text"] = item_catalog["competence_text"].fillna(
    item_catalog["competence_id"]
)

item_catalog["competence_name"] = item_catalog["competence_name"].fillna("")
item_catalog["competence_code"] = item_catalog["competence_code"].fillna("")

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1, 2),
    min_df=1,
    lowercase=True
)

item_text_matrix = vectorizer.fit_transform(
    item_catalog["competence_text"]
)

item_text_matrix = normalize(item_text_matrix)

print("item_text_matrix:", item_text_matrix.shape)


item_text_matrix: (92, 1028)


Вспомогательные словари

Нужно быстро получать:

* индекс пользователя;
* индекс компетенции;
* текущее состояние user × competence;
* историю пользователя.

In [ ]:
user_to_idx = {
    user_id: idx
    for idx, user_id in enumerate(user_index.astype(str))
}

comp_to_idx = {
    comp_id: idx
    for idx, comp_id in enumerate(comp_index.astype(str))
}

In [ ]:
user_comp_state = (
    model_df
    .set_index(["user_id", "competence_id"])
    [[
        "probability",
        "need",
        "evidence_count",
        "task_attempts",
        "total_experience",
        "experience_weight"
    ]]
    .to_dict("index")
)

In [ ]:
def minmax(x):
    x = np.asarray(x, dtype=float)

    x = np.nan_to_num(
        x,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    mn = x.min()
    mx = x.max()

    if mx - mn < 1e-12:
        return np.zeros_like(x)

    return (x - mn) / (mx - mn)

In [ ]:
def get_user_history(user_id):
    return model_df[
        model_df["user_id"].astype(str) == str(user_id)
    ].copy()

Текстовый профиль слабых зон пользователя

Для каждого пользователя строим текстовый профиль его слабых мест.

* probability низкая
* need высокий
* есть evidence_count или task_attempts

In [ ]:
def build_user_weak_text_profile(user_id):
    hist = get_user_history(user_id)

    if hist.empty:
        return None

    hist["comp_idx"] = hist["competence_id"].map(comp_to_idx)

    hist = hist.dropna(
        subset=["comp_idx"]
    ).copy()

    hist["comp_idx"] = hist["comp_idx"].astype(int)

    weights = (
        hist["need"].values
        * (1 + hist["experience_weight"].values)
    )

    if np.isclose(weights.sum(), 0):
        weights = 1 + hist["experience_weight"].values

    profile = (
        weights.reshape(1, -1)
        @ item_text_matrix[hist["comp_idx"].values]
    )

    profile = normalize(profile)

    return profile

Гибридная рекомендация компетенций

Он использует 4 сигнала:

* ALS-score — опыт похожих пользователей;
* text-score — похожесть на слабые зоны пользователя;
* need-score — насколько компетенция плохо освоена;
* experience-score — насколько по этой компетенции уже есть опыт.


final_score =
    0.45 × ALS_score + 0.25 × text_score + 0.25 × need_score + 0.05 × experience_score

В прошлый раз я исключал все известные компетенции. Здесь так делать нельзя. Если пользователь уже видел компетенцию, но плохо её знает, её как раз надо рекомендовать.

Поэтому исключаем только те компетенции, которые уже хорошо освоены.

In [ ]:
def recommend_competences_hybrid(
    user_id,
    top_n=30,
    w_als = 0.35,
    w_text = 0.25,
    w_need = 0.30,
    w_experience = 0.10,
    mastery_threshold=0.75,
    min_evidence_for_mastery=3,
    keep_seen_weak=True
):
    user_id = str(user_id)

    n_items = len(comp_index)

    # 1. ALS-score
    if user_id in user_to_idx:
        user_idx = user_to_idx[user_id]

        user_vec = als_model.user_factors[user_idx]

        als_scores = (
            als_model.item_factors
            @ user_vec
        )
    else:
        als_scores = np.zeros(n_items)

    # 2. Text-score
    user_profile = build_user_weak_text_profile(user_id)

    if user_profile is not None:
        text_scores_raw = item_text_matrix @ user_profile.T

        if hasattr(text_scores_raw, "toarray"):
            text_scores = text_scores_raw.toarray().ravel()
        else:
            text_scores = np.asarray(text_scores_raw).ravel()
    else:
        text_scores = np.zeros(n_items)

    need_scores = np.zeros(n_items)
    experience_scores = np.zeros(n_items)

    current_mastery = np.full(n_items, np.nan)
    current_evidence = np.zeros(n_items)
    current_task_attempts = np.zeros(n_items)

    for comp_id, idx in comp_to_idx.items():
        state = user_comp_state.get(
            (user_id, comp_id)
        )

        if state is None:

            need_scores[idx] = 0.35
            experience_scores[idx] = 0.0
        else:
            current_mastery[idx] = state["probability"]
            current_evidence[idx] = state["evidence_count"]
            current_task_attempts[idx] = state["task_attempts"]

            need_scores[idx] = state["need"]
            experience_scores[idx] = state["experience_weight"]

    final_scores = (
        w_als * minmax(als_scores)
        + w_text * minmax(text_scores)
        + w_need * minmax(need_scores)
        + w_experience * minmax(experience_scores)
    )

    rec = pd.DataFrame({
        "user_id": user_id,
        "competence_id": comp_index.astype(str),
        "competence_code": item_catalog["competence_code"].values,
        "competence_name": item_catalog["competence_name"].values,

        "als_score_raw": als_scores,
        "text_score_raw": text_scores,
        "need_score_raw": need_scores,
        "experience_score_raw": experience_scores,

        "als_score": minmax(als_scores),
        "text_score": minmax(text_scores),
        "need_score": minmax(need_scores),
        "experience_score": minmax(experience_scores),

        "final_score": final_scores,

        "current_mastery": current_mastery,
        "current_evidence": current_evidence,
        "current_task_attempts": current_task_attempts
    })

    rec = rec[
        (rec["current_evidence"] >= 2)
        | (rec["current_mastery"].isna())
    ].copy()

    mastered_mask = (
        rec["current_mastery"].notna()
        & (rec["current_mastery"] >= mastery_threshold)
        & (rec["current_evidence"] >= min_evidence_for_mastery)
    )

    if keep_seen_weak:
        rec = rec[~mastered_mask].copy()
    else:
        seen_mask = rec["current_mastery"].notna()
        rec = rec[~seen_mask & ~mastered_mask].copy()

    conditions = [
        rec["need_score"] >= 0.70,
        rec["text_score"] >= 0.70,
        rec["als_score"] >= 0.70
    ]

    choices = [
        "низкое владение по UserCompetence",
        "похожа на слабые зоны пользователя по тексту",
        "рекомендуется ALS по опыту похожих пользователей"
    ]

    rec["main_reason"] = np.select(
        conditions,
        choices,
        default="смешанный гибридный сигнал"
    )

    rec = rec.sort_values(
        "final_score",
        ascending=False
    ).reset_index(drop=True)

    rec["rank"] = np.arange(1, len(rec) + 1)

    return rec.head(top_n)

Пример рекомендации компетенций

In [ ]:
example_user = str(user_index[0])

example_comp_rec = recommend_competences_hybrid(
    user_id=example_user,
    top_n=30,
    mastery_threshold=0.75,
    keep_seen_weak=True
)

print("example_user:", example_user)

display(
    example_comp_rec[[
        "rank",
        "competence_code",
        "competence_name",
        "final_score",
        "als_score",
        "text_score",
        "need_score",
        "experience_score",
        "current_mastery",
        "current_evidence",
        "current_task_attempts",
        "main_reason"
    ]]
)

example_user: 04faac8a-55ba-45e8-b22b-3defa5512390


,rank,competence_code,competence_name,final_score,als_score,text_score,need_score,experience_score,current_mastery,current_evidence,current_task_attempts,main_reason
0,1,9.2.1.,Умеет интерпретировать условие «значение выраж...,0.914402,0.995675,0.999422,0.916399,0.411408,0.115756,2.0,2.0,низкое владение по UserCompetence
1,2,9.4.4.,Умеет интерпретировать условие «значение выраж...,0.893184,0.990205,0.922206,0.916399,0.411408,0.115756,2.0,2.0,низкое владение по UserCompetence
2,3,4.1.2,Умеет определять число благоприятных исходов m...,0.860426,0.998183,1.000000,0.704399,0.497418,0.320316,3.0,3.0,низкое владение по UserCompetence
3,4,6.16.1,Умеет представлять числа и дроби в виде степен...,0.845315,0.994944,0.724097,0.916399,0.411408,0.115756,2.0,2.0,низкое владение по UserCompetence
4,5,7.0.0,Вычисления и преобразования,0.833914,0.996752,0.575639,1.000000,0.411408,0.035088,2.0,2.0,низкое владение по UserCompetence
5,6,9.6.6.,Умеет учитывать ограничение на переменную по с...,0.831365,0.990245,0.674873,0.916399,0.411408,0.115756,2.0,2.0,низкое владение по UserCompetence
6,7,9.1.6.,Умеет выполнять численные вычисления по получе...,0.777083,0.991330,0.676225,0.704399,0.497418,0.320316,3.0,3.0,низкое владение по UserCompetence
7,8,6.15.2,Умеет приравнивать показатели степени при равн...,0.751169,0.997628,0.563749,0.704399,0.497418,0.320316,3.0,3.0,низкое владение по UserCompetence
8,9,9.1.3.,Умеет подставлять известные значения (числа ил...,0.741236,0.994029,0.529058,0.704399,0.497418,0.320316,3.0,3.0,низкое владение по UserCompetence
9,10,4.1.1,Умеет определять число всех равновозможных исх...,0.724561,0.997133,0.816404,0.384325,0.561659,0.629160,4.0,4.0,похожа на слабые зоны пользователя по тексту


Рекомендации компетенций для всех пользователей

In [ ]:
all_comp_recs = []

for user_id in tqdm(user_index.astype(str)):
    rec = recommend_competences_hybrid(
        user_id=user_id,
        top_n=30,
        mastery_threshold=0.75,
        keep_seen_weak=True
    )

    all_comp_recs.append(rec)

hybrid_competence_rec = pd.concat(
    all_comp_recs,
    ignore_index=True
)

display(hybrid_competence_rec.head())

100%|██████████| 30/30 [00:00<00:00, 121.50it/s]


,user_id,competence_id,competence_code,competence_name,als_score_raw,text_score_raw,need_score_raw,experience_score_raw,als_score,text_score,need_score,experience_score,final_score,current_mastery,current_evidence,current_task_attempts,main_reason,rank
0,04faac8a-55ba-45e8-b22b-3defa5512390,6aafe9b9-9d86-442a-9054-0372b4ff08d6,9.2.1.,Умеет интерпретировать условие «значение выраж...,0.999106,0.371418,0.884244,1.609438,0.995675,0.999422,0.916399,0.411408,0.914402,0.115756,2.0,2.0,низкое владение по UserCompetence,1
1,04faac8a-55ba-45e8-b22b-3defa5512390,b4cd88be-c890-4d73-9c8a-2071f67c2af2,9.4.4.,Умеет интерпретировать условие «значение выраж...,0.993368,0.342722,0.884244,1.609438,0.990205,0.922206,0.916399,0.411408,0.893184,0.115756,2.0,2.0,низкое владение по UserCompetence,2
2,04faac8a-55ba-45e8-b22b-3defa5512390,5067001a-a115-4624-823b-4de68176f569,4.1.2,Умеет определять число благоприятных исходов m...,1.001737,0.371633,0.679684,1.945910,0.998183,1.000000,0.704399,0.497418,0.860426,0.320316,3.0,3.0,низкое владение по UserCompetence,3
3,04faac8a-55ba-45e8-b22b-3defa5512390,118d8585-a00f-46f7-be45-0a39fc33f401,6.16.1,Умеет представлять числа и дроби в виде степен...,0.998340,0.269098,0.884244,1.609438,0.994944,0.724097,0.916399,0.411408,0.845315,0.115756,2.0,2.0,низкое владение по UserCompetence,4
4,04faac8a-55ba-45e8-b22b-3defa5512390,1024e3ec-6786-4059-84fe-3ae001527444,7.0.0,Вычисления и преобразования,1.000236,0.213926,0.964912,1.609438,0.996752,0.575639,1.000000,0.411408,0.833914,0.035088,2.0,2.0,низкое владение по UserCompetence,5


In [ ]:
hybrid_competence_rec.to_csv(
    OUTPUT_DIR / "hybrid_competence_recommendations.csv",
    index=False,
    encoding="utf-8-sig"
)

Переход от компетенций к микромодулям

coverage_score — какую долю компетенций микромодуля покрывает рекомендация;


jaccard_score — пересечение рекомендованных компетенций и компетенций микромодуля;


weighted_coverage_score — то же самое, но с учётом final_score компетенций.

In [ ]:
USE_ONLY_MAIN_MICROMODULES = False

In [ ]:
def prepare_micro_source(micro_df):
    m = micro_df.copy()

    for col in [
        "node_id",
        "node_name",
        "competence_id",
        "course_description",
        "task_number"
    ]:
        if col in m.columns:
            m[col] = m[col].astype(str)

    if USE_ONLY_MAIN_MICROMODULES and "importance" in m.columns:
        filtered = m[
            pd.to_numeric(
                m["importance"],
                errors="coerce"
            ).fillna(1.0) >= 1.0
        ].copy()

        if not filtered.empty:
            m = filtered

    return m

In [ ]:
micro_source = prepare_micro_source(micro_df)

micro_comp_dict = (
    micro_source
    .groupby("node_id")["competence_id"]
    .apply(lambda x: set(x.astype(str)))
    .to_dict()
)

micro_meta = (
    micro_source
    .groupby("node_id", as_index=False)
    .agg(
        node_name=("node_name", "first"),
        task_number=("task_number", "first"),
        course_description=("course_description", "first"),
        micro_competences_count=("competence_id", "nunique")
    )
)

micro_meta_dict = (
    micro_meta
    .set_index("node_id")
    .to_dict("index")
)

comp_name_dict = (
    comp_catalog[["competence_id", "competence_name"]]
    .drop_duplicates("competence_id")
    .set_index("competence_id")["competence_name"]
    .to_dict()
)

print("Микромодулей:", len(micro_comp_dict))
display(micro_meta.head())

Микромодулей: 192


,node_id,node_name,task_number,course_description,micro_competences_count
0,0226f63b-043e-43c1-9db1-5157c951a514,19. Нахождение экстремумы функции с помощью пр...,12,Значения функций,6
1,02527317-f5cd-4d6e-84d0-3a55505645d5,22. Классическая вероятность в условном простр...,4,Теория Вероятностей,7
2,05168121-2cbb-4706-9ff0-8f1828b1fb9c,3. Без производной - Определение экстремума ло...,12,Значения функций,4
3,065ed65d-be42-438d-aa3c-06d66fc0a35d,46. Нахождение наибольшего/наименьшего значени...,12,Значения функций,11
4,078d00da-78ac-4fea-880c-d45946b31c24,24. Решение логарифмических уравнений с дробны...,6,Простейшие уравнения,2


In [ ]:
def jaccard(a, b):
    a = set(a)
    b = set(b)

    union = a | b

    if len(union) == 0:
        return 0

    return len(a & b) / len(union)

Функция рекомендации микромодулей

In [ ]:
def recommend_micro_modules(
    user_id,
    rec_df,
    top_comp=30,
    top_micro=10
):
    user_id = str(user_id)

    user_rec = (
        rec_df[
            rec_df["user_id"].astype(str) == user_id
        ]
        .sort_values("rank")
        .head(top_comp)
        .copy()
    )

    if user_rec.empty:
        return pd.DataFrame()

    rec_competences = (
        user_rec["competence_id"]
        .astype(str)
        .tolist()
    )

    rec_comp_set = set(rec_competences)

    rec_weight = dict(
        zip(
            user_rec["competence_id"].astype(str),
            user_rec["final_score"].astype(float)
        )
    )

    rows = []

    for node_id, node_comp_set in micro_comp_dict.items():
        matched = rec_comp_set & node_comp_set

        if len(matched) == 0:
            continue

        weighted_sum = sum(
            rec_weight.get(comp_id, 0)
            for comp_id in matched
        )

        coverage_score = (
            len(matched)
            / max(len(node_comp_set), 1)
        )

        weighted_coverage_score = (
            weighted_sum
            / max(len(node_comp_set), 1)
        )

        meta = micro_meta_dict.get(node_id, {})

        matched_names = [
            comp_name_dict.get(comp_id, comp_id)
            for comp_id in matched
        ]

        rows.append({
            "user_id": user_id,
            "node_id": node_id,
            "node_name": meta.get("node_name", ""),
            "task_number": meta.get("task_number", ""),
            "course_description": meta.get("course_description", ""),
            "micro_competences_count": meta.get(
                "micro_competences_count",
                len(node_comp_set)
            ),
            "matched_competences_count": len(matched),
            "coverage_score": coverage_score,
            "jaccard_score": jaccard(rec_comp_set, node_comp_set),
            "weighted_coverage_score": weighted_coverage_score,
            "matched_competence_ids": " | ".join(sorted(matched)),
            "matched_competence_names": " | ".join(
                sorted(set(matched_names))
            )
        })

    result = pd.DataFrame(rows)

    if result.empty:
        return result

    result = result.sort_values(
        [
            "weighted_coverage_score",
            "coverage_score",
            "jaccard_score"
        ],
        ascending=False
    ).reset_index(drop=True)

    result["rank"] = np.arange(1, len(result) + 1)

    return result.head(top_micro)

In [ ]:
example_micro_rec = recommend_micro_modules(
    user_id=example_user,
    rec_df=hybrid_competence_rec,
    top_comp=30,
    top_micro=10
)

print("example_user:", example_user)

display(
    example_micro_rec[[
        "rank",
        "task_number",
        "node_name",
        "course_description",
        "weighted_coverage_score",
        "coverage_score",
        "jaccard_score",
        "matched_competences_count",
        "micro_competences_count",
        "matched_competence_names"
    ]]
)

example_user: 04faac8a-55ba-45e8-b22b-3defa5512390


,rank,task_number,node_name,course_description,weighted_coverage_score,coverage_score,jaccard_score,matched_competences_count,micro_competences_count,matched_competence_names
0,1,7,Номер 7 ЕГЭ,Вычисления,0.833914,1.0,0.033333,1,1,Вычисления и преобразования
1,2,4,11. Классическая вероятность попадания в одну ...,Теория Вероятностей,0.715647,1.0,0.133333,4,4,Умеет вычислять вероятность события по формуле...
2,3,4,1. Классическое определение вероятности: равно...,Теория Вероятностей,0.715647,1.0,0.133333,4,4,Умеет вычислять вероятность события по формуле...
3,4,5,Номер 5 ЕГЭ,Усложненная Теория Вероятностей,0.620327,1.0,0.033333,1,1,Сложные задачи по теории вероятностей
4,5,2,Номер 2 ЕГЭ,Векторы,0.579458,1.0,0.033333,1,1,Векторы
5,6,4,"12. Жеребьевка очередности: вероятность на ""фи...",Теория Вероятностей,0.572518,0.8,0.129032,4,5,Умеет вычислять вероятность события по формуле...
6,7,4,5. Классическая вероятность при вычислении чис...,Теория Вероятностей,0.572518,0.8,0.129032,4,5,Умеет вычислять вероятность события по формуле...
7,8,4,7. Оценка отклонения частоты события от теорет...,Теория Вероятностей,0.572518,0.8,0.129032,4,5,Умеет вычислять вероятность события по формуле...
8,9,4,17. Игральные кости: классическая вероятность ...,Теория Вероятностей,0.572518,0.8,0.129032,4,5,Умеет вычислять вероятность события по формуле...
9,10,4,6. Вероятность совместного попадания двух учас...,Теория Вероятностей,0.572518,0.8,0.129032,4,5,Умеет вычислять вероятность события по формуле...


In [ ]:
all_micro_recs = []

for user_id in tqdm(user_index.astype(str)):
    rec = recommend_micro_modules(
        user_id=user_id,
        rec_df=hybrid_competence_rec,
        top_comp=30,
        top_micro=10
    )

    if not rec.empty:
        all_micro_recs.append(rec)

hybrid_micro_rec = pd.concat(
    all_micro_recs,
    ignore_index=True
)

display(hybrid_micro_rec.head(200))

100%|██████████| 30/30 [00:00<00:00, 208.79it/s]


,user_id,node_id,node_name,task_number,course_description,micro_competences_count,matched_competences_count,coverage_score,jaccard_score,weighted_coverage_score,matched_competence_ids,matched_competence_names,rank
0,04faac8a-55ba-45e8-b22b-3defa5512390,599e4cef-1b6a-4027-a8dc-5fad3647a149,Номер 7 ЕГЭ,7,Вычисления,1,1,1.000000,0.033333,0.833914,1024e3ec-6786-4059-84fe-3ae001527444,Вычисления и преобразования,1
1,04faac8a-55ba-45e8-b22b-3defa5512390,0c25e3af-d751-4490-b181-de7ab2cc1660,11. Классическая вероятность попадания в одну ...,4,Теория Вероятностей,4,4,1.000000,0.133333,0.715647,2e7f87eb-5a14-4b20-99f4-7efc507e829b | 5067001...,Умеет вычислять вероятность события по формуле...,2
2,04faac8a-55ba-45e8-b22b-3defa5512390,d0700977-3f66-49e5-b561-fbb92f53c794,1. Классическое определение вероятности: равно...,4,Теория Вероятностей,4,4,1.000000,0.133333,0.715647,2e7f87eb-5a14-4b20-99f4-7efc507e829b | 5067001...,Умеет вычислять вероятность события по формуле...,3
3,04faac8a-55ba-45e8-b22b-3defa5512390,b25788c9-58ae-4659-9527-b8e1e748b274,Номер 5 ЕГЭ,5,Усложненная Теория Вероятностей,1,1,1.000000,0.033333,0.620327,5cf7d4dd-1c3f-4fb7-821c-6d4aa6df09aa,Сложные задачи по теории вероятностей,4
4,04faac8a-55ba-45e8-b22b-3defa5512390,25570988-2dad-48c6-9974-af6a4a36c3f6,Номер 2 ЕГЭ,2,Векторы,1,1,1.000000,0.033333,0.579458,e28b7ebb-d852-472a-8b46-99621bf850d1,Векторы,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,94ea1dd8-1461-4345-b65a-4a7358256e5f,726edb0e-fcd3-438b-8750-6dc7ad059eb1,Номер 10 ЕГЭ,10,Текстовые задачи,1,1,1.000000,0.033333,0.670676,e0bf6fb3-c7cf-47af-8f34-aa79a7837321,Текстовые задачи,6
196,94ea1dd8-1461-4345-b65a-4a7358256e5f,b96468f6-9f2a-40ec-b145-4285ae98258b,43. Нахождение наибольшего/наименьшего значени...,12,Значения функций,7,5,0.714286,0.156250,0.624277,60885780-2c99-43e9-95c5-2edf4a056993 | 742e39f...,Умеет выбрать одну из граничных точек отрезка ...,7
197,94ea1dd8-1461-4345-b65a-4a7358256e5f,c2fab005-cf31-4dbf-9002-ffc5700f28d5,10. Расчет предельного времени/значения по фор...,9,Прикладные задачи,8,7,0.875000,0.225806,0.583438,3acd7099-5c7c-471e-a65d-de2a85de0680 | 4462b5b...,Умеет выбрать меньший/больший из корней уравне...,8
198,94ea1dd8-1461-4345-b65a-4a7358256e5f,afb18c2e-80f9-498a-83ad-268cde279bcf,8. Расчет времени до полного опустошения бака ...,9,Прикладные задачи,6,5,0.833333,0.161290,0.552311,3acd7099-5c7c-471e-a65d-de2a85de0680 | 4462b5b...,Умеет выполнять численные вычисления по получе...,9


In [ ]:
hybrid_micro_rec.to_csv(
    OUTPUT_DIR / "hybrid_micro_module_recommendations.csv",
    index=False,
    encoding="utf-8-sig"
)

Сводка по пользователям

* сколько задач решил;
* какой correct rate;
* сколько компетенций оценено;
* средний уровень владения;
* средняя потребность;
* сколько всего evidence.

In [ ]:
comp_user_agg = (
    model_df
    .groupby("user_id", as_index=False)
    .agg(
        assessed_competences=("competence_id", "nunique"),
        avg_mastery=("probability", "mean"),
        avg_need=("need", "mean"),
        total_evidence=("evidence_count", "sum"),
        avg_evidence=("evidence_count", "mean")
    )
)

In [ ]:
user_profile_summary = (
    pd.DataFrame({
        "user_id": user_index.astype(str)
    })
    .merge(
        user_summary,
        on="user_id",
        how="left"
    )
    .merge(
        comp_user_agg,
        on="user_id",
        how="left"
    )
)

display(user_profile_summary.head())

,user_id,solved_tasks,unique_tasks,unique_nodes,task_correct_rate,first_task_time,last_task_time,total_practices,kim_count,block_count,avg_test_score,avg_total_score,first_practice_time,last_practice_time,assessed_competences,avg_mastery,avg_need,total_evidence,avg_evidence
0,04faac8a-55ba-45e8-b22b-3defa5512390,64.0,58.0,17.0,0.953125,2026-02-25 17:41:51.998349+00:00,2026-03-22 09:40:01.786033+00:00,9,4,5,38.555556,6.777778,2026-02-25 17:41:51.998349+00:00,2026-03-22 09:40:01.786033+00:00,32,0.288507,0.711493,117,3.656250
1,0e24ff0e-c58a-4a6c-b054-e80df6607b99,14.0,14.0,14.0,0.785714,2026-03-11 20:18:02.450819+00:00,2026-03-11 20:18:02.450819+00:00,1,1,0,76.000000,15.000000,2026-03-11 20:18:02.450819+00:00,2026-03-11 20:18:02.450819+00:00,30,0.036652,0.963348,31,1.033333
2,17993c65-1f0e-4f0a-b458-a7c688b99ea0,212.0,177.0,46.0,0.622642,2026-02-20 06:17:00.025055+00:00,2026-04-09 16:45:52.223790+00:00,19,10,9,39.052632,6.947368,2026-02-20 06:17:00.025055+00:00,2026-04-09 16:45:52.223790+00:00,76,0.232077,0.767923,512,6.736842
3,2313e3e0-a834-4dfd-a62e-da6bf6bab116,9.0,9.0,8.0,0.222222,NaT,NaT,3,3,0,3.666667,0.666667,2026-01-29 17:26:14.122537+00:00,2026-01-29 17:39:01.419094+00:00,5,0.008094,0.991906,7,1.400000
4,363f7164-3a52-43aa-8b9b-b1677ed218ec,216.0,184.0,55.0,0.750000,2026-02-22 14:49:19.246889+00:00,2026-04-05 07:56:28.460589+00:00,17,5,12,60.352941,11.176471,2026-02-22 14:49:19.246889+00:00,2026-04-05 07:56:28.460589+00:00,64,0.457486,0.542514,483,7.546875


In [ ]:
user_profile_summary.to_csv(
    OUTPUT_DIR / "user_profile_summary.csv",
    index=False,
    encoding="utf-8-sig"
)